# Notebook 25 — Geodesic Transport and Residual Flow

This notebook turns the residual universality manifold from Notebook 24 into a transport geometry.

It computes:

- residual manifold kNN graph
- geodesic / shortest-path distances
- family-to-family transport paths
- density- and confidence-weighted transport costs
- bottleneck regions
- bridgeability scores
- transport vector-field samples
- downloadable figure/result pack

The notebook is designed to run in Google Colab from the repo root, or from `notebooks/` inside the repo.

In [ ]:
# Notebook 25 setup

import os
import json
import math
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

warnings.filterwarnings("ignore")

try:
    import networkx as nx
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx"])
    import networkx as nx

try:
    from scipy.interpolate import griddata
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy.interpolate import griddata

plt.rcParams.update({
    "figure.figsize": (10, 7),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## 1. Resolve repo paths

This cell avoids the recurring Colab path issue.

It checks:

1. current directory,
2. parent directory,
3. `/content/residue-manifold-learning`,
4. cloned repo path if available.

It then creates `figures/`, `results/`, and `exports/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_NAME = "residue-manifold-learning"
REPO_URL = "https://github.com/thinkthoughts/residue-manifold-learning.git"


def has_useful_results(root: Path) -> bool:
    """Return True when a candidate repo root has result files, not merely an empty results dir."""
    root = Path(root)
    results = root / "results"
    if not results.exists():
        return False
    return any(results.glob("*.csv")) or any(results.glob("*.json"))


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        Path("/content") / REPO_NAME,
        Path("/content"),
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]

    # Prefer roots with actual result files.
    for c in candidates:
        if has_useful_results(c):
            return c.resolve()

    # Then accept a likely repo root, even if results has not been generated yet.
    for c in candidates:
        if (c / "notebooks").exists() or (c / ".git").exists():
            return c.resolve()

    return cwd


def maybe_clone_repo() -> Path:
    """Clone repo into /content when Colab starts from an empty /content workspace."""
    target = Path("/content") / REPO_NAME
    if target.exists():
        return target.resolve()

    try:
        print(f"No local repo checkout found. Cloning {REPO_URL} ...")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
        return target.resolve()
    except Exception as exc:
        print("Clone skipped or unavailable:", exc)
        return Path.cwd().resolve()


REPO_ROOT = find_repo_root()

# In Colab, Path.cwd() is often /content and results/ is empty. Try cloning before falling back.
if not has_useful_results(REPO_ROOT):
    cloned = maybe_clone_repo()
    if has_useful_results(cloned) or (cloned / "notebooks").exists():
        REPO_ROOT = cloned

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
EXPORTS_DIR = REPO_ROOT / "exports"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("cwd:", Path.cwd().resolve())
print("repo root:", REPO_ROOT)
print("results dir:", RESULTS_DIR)
print("available result files:", sorted(p.name for p in RESULTS_DIR.glob("*"))[:50])


## 2. Load known residual manifold data

Notebook 25 accepts several possible upstream files.

Preferred files from Notebook 24 / 20 / 18:

- `residual_universality_embedding.csv`
- `residual_pca_embedding.csv`
- `residual_classification_feature_matrix.csv`
- `residual_geometry_features.csv`

It standardizes columns into:

- `family`
- `N`
- `PC1`
- `PC2`

In [ ]:
def read_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f"loaded: {p}")
            return pd.read_csv(p), p
    return None, None


def candidate_result_dirs():
    dirs = []
    for root in [
        REPO_ROOT,
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path("/content") / REPO_NAME,
        Path("/content"),
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]:
        d = Path(root) / "results"
        if d.exists() and d not in dirs:
            dirs.append(d)
    return dirs


candidate_names = [
    "residual_universality_embedding.csv",
    "residual_pca_embedding.csv",
    "residual_classification_feature_matrix.csv",
    "residual_geometry_features.csv",
    "residual_pca_embedding_v2.csv",
    "residual_manifold_embedding.csv",
    "residual_density_known_points.csv",
]

known_candidates = []
for d in candidate_result_dirs():
    for name in candidate_names:
        known_candidates.append(d / name)

known_raw, known_path = read_first_existing(known_candidates)


def synthesize_known_residual_embedding(out_path: Path) -> pd.DataFrame:
    """Deterministic fallback so Notebook 25 still runs when Colab has no repo outputs loaded.

    This is not a replacement for Notebook 18/20/21/24 data. It is a compact continuity scaffold
    that preserves the same columns and family geometry needed to test Notebook 25 end-to-end.
    """
    rows = []
    coords = {
        "ring lattice":       [(16, -0.55,  1.45), (32, -0.20,  0.10), (64, -1.15, -0.45), (128, -1.55, -1.05)],
        "small world":        [(16, -0.30,  3.45), (32, -3.05,  1.90), (64, -3.10, -2.05), (128, -2.95, -1.60)],
        "Erdős–Rényi":        [(16,  0.25,  0.90), (32,  0.10,  0.05), (64, -1.10, -0.45), (128, -1.45, -1.00)],
        "scale free":         [(16,  2.70,  0.30), (32,  1.45, -0.25), (64,  1.30, -0.70), (128,  0.55, -0.15)],
        "modular clustered":  [(16,  4.60, -0.65), (32,  3.65, -0.35), (64,  3.25, -0.10), (128,  3.05,  0.00)],
    }
    for fam, pts in coords.items():
        for N, pc1, pc2 in pts:
            rows.append({"family": fam, "N": N, "PC1": pc1, "PC2": pc2, "source": "synthetic_fallback"})
    df = pd.DataFrame(rows)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return df


if known_raw is None:
    available_by_dir = {str(d): sorted(p.name for p in d.glob("*")) for d in candidate_result_dirs()}
    print("No known residual manifold file found in candidate result dirs.")
    print(json.dumps(available_by_dir, indent=2)[:3000])
    print("\nUsing deterministic fallback data so the notebook can run end-to-end.")
    fallback_path = RESULTS_DIR / "25_synthetic_known_residual_embedding.csv"
    known_raw = synthesize_known_residual_embedding(fallback_path)
    known_path = fallback_path

known_raw.head()


In [ ]:
def normalize_known_embedding(df):
    df = df.copy()

    # family/topology column
    if "family" not in df.columns:
        for alt in ["topology", "known_family", "label", "graph_family"]:
            if alt in df.columns:
                df["family"] = df[alt]
                break

    # graph size column
    if "N" not in df.columns:
        for alt in ["n_modules", "graph_size", "size", "n", "num_nodes"]:
            if alt in df.columns:
                df["N"] = df[alt]
                break

    # coordinate columns
    if "PC1" not in df.columns:
        for alt in ["pc1", "x", "coord1", "manifold_coordinate_1", "residual manifold coordinate 1"]:
            if alt in df.columns:
                df["PC1"] = df[alt]
                break

    if "PC2" not in df.columns:
        for alt in ["pc2", "y", "coord2", "manifold_coordinate_2", "residual manifold coordinate 2"]:
            if alt in df.columns:
                df["PC2"] = df[alt]
                break

    # If no PC coordinates but enough numeric features, compute PCA.
    if not {"PC1", "PC2"}.issubset(df.columns):
        exclude = {"family", "topology", "label", "N", "n", "n_modules", "graph_size", "size"}
        numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
        if len(numeric_cols) >= 2:
            X = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
            X = X.fillna(X.median(numeric_only=True))
            Xs = StandardScaler().fit_transform(X)
            pcs = PCA(n_components=2, random_state=0).fit_transform(Xs)
            df["PC1"] = pcs[:, 0]
            df["PC2"] = pcs[:, 1]
            print("computed PC1/PC2 from numeric feature columns:", numeric_cols)
        else:
            raise ValueError("Could not find PC1/PC2 columns or enough numeric features to compute PCA.")

    if "family" not in df.columns:
        raise ValueError("Could not infer family/topology column.")

    if "N" not in df.columns:
        # fallback: order within family
        df["N"] = df.groupby("family").cumcount().map(lambda i: [16, 32, 64, 128][i % 4])

    df = df[["family", "N", "PC1", "PC2"] + [c for c in df.columns if c not in ["family", "N", "PC1", "PC2"]]].copy()
    df["family"] = df["family"].astype(str)
    df["N"] = pd.to_numeric(df["N"], errors="coerce")
    df["PC1"] = pd.to_numeric(df["PC1"], errors="coerce")
    df["PC2"] = pd.to_numeric(df["PC2"], errors="coerce")
    df = df.dropna(subset=["family", "N", "PC1", "PC2"]).reset_index(drop=True)
    return df

known = normalize_known_embedding(known_raw)
known.to_csv(RESULTS_DIR / "25_known_residual_embedding_normalized.csv", index=False)

print("known rows:", len(known))
print(known[["family", "N", "PC1", "PC2"]].head())
known.groupby("family").size()

## 3. Estimate density, confidence, and local curvature proxies

Notebook 25 can run even if Notebook 24 density fields are unavailable.

It estimates:

- density support from kNN distances,
- boundary confidence from nearest-centroid margin,
- local curvature from family trajectories.

In [ ]:
families = sorted(known["family"].unique())
coords = known[["PC1", "PC2"]].to_numpy()

# family centroids
centroids = known.groupby("family")[["PC1", "PC2"]].mean()
centroid_points = centroids.to_numpy()
centroid_names = list(centroids.index)

# density proxy from kNN distance
k_density = min(5, max(2, len(known)-1))
nbrs = NearestNeighbors(n_neighbors=k_density).fit(coords)
distances, indices = nbrs.kneighbors(coords)
mean_knn_dist = distances[:, 1:].mean(axis=1)
density = 1 / (mean_knn_dist + 1e-9)
density = (density - density.min()) / (density.max() - density.min() + 1e-12)

# confidence proxy from nearest and second-nearest family centroid
Dcent = pairwise_distances(coords, centroid_points)
nearest = np.argsort(Dcent, axis=1)
d1 = Dcent[np.arange(len(coords)), nearest[:, 0]]
d2 = Dcent[np.arange(len(coords)), nearest[:, 1]] if len(centroid_names) > 1 else d1 + 1
margin = (d2 - d1) / (d2 + 1e-9)
confidence = np.clip(margin, 0, 1)

known["density_proxy"] = density
known["confidence_proxy"] = confidence
known["nearest_centroid_family"] = [centroid_names[i] for i in nearest[:, 0]]

# trajectory curvature proxy
known["local_curvature"] = 0.0
for fam, sub in known.sort_values("N").groupby("family"):
    idx = sub.index.to_list()
    pts = sub[["PC1", "PC2"]].to_numpy()
    if len(pts) >= 3:
        curv = np.zeros(len(pts))
        for i in range(1, len(pts)-1):
            curv[i] = np.linalg.norm(pts[i+1] - 2*pts[i] + pts[i-1])
        curv[0] = curv[1]
        curv[-1] = curv[-2]
    else:
        curv = np.zeros(len(pts))
    known.loc[idx, "local_curvature"] = curv

# normalize curvature for weighting
c = known["local_curvature"].to_numpy()
known["curvature_proxy"] = (c - c.min()) / (c.max() - c.min() + 1e-12)

known.to_csv(RESULTS_DIR / "25_known_residual_transport_features.csv", index=False)
known.head()

## 4. Build residual manifold kNN graph

Edge cost combines:

\[
cost_{ij}
=
d_{ij}
(1 + \bar{\kappa}_{ij})
(1 + (1 - \bar{C}_{ij}))
(1 + (1 - \bar{\rho}_{ij}))
\]

So transport through:
- low density,
- low confidence,
- high curvature

becomes more expensive.

In [ ]:
k_graph = min(5, max(2, len(known)-1))
nn = NearestNeighbors(n_neighbors=k_graph).fit(coords)
dist, ind = nn.kneighbors(coords)

G = nx.Graph()

for i, row in known.iterrows():
    G.add_node(
        i,
        family=row["family"],
        N=float(row["N"]),
        PC1=float(row["PC1"]),
        PC2=float(row["PC2"]),
        density=float(row["density_proxy"]),
        confidence=float(row["confidence_proxy"]),
        curvature=float(row["curvature_proxy"]),
    )

edge_rows = []
for i in range(len(known)):
    for d, j in zip(dist[i, 1:], ind[i, 1:]):
        if i == j:
            continue
        rho = 0.5 * (known.loc[i, "density_proxy"] + known.loc[j, "density_proxy"])
        conf = 0.5 * (known.loc[i, "confidence_proxy"] + known.loc[j, "confidence_proxy"])
        curv = 0.5 * (known.loc[i, "curvature_proxy"] + known.loc[j, "curvature_proxy"])

        cost = float(d * (1 + curv) * (1 + (1-conf)) * (1 + (1-rho)))
        G.add_edge(i, int(j), distance=float(d), weight=cost, density=rho, confidence=conf, curvature=curv)

        edge_rows.append({
            "source": i,
            "target": int(j),
            "euclidean_distance": float(d),
            "transport_cost": cost,
            "edge_density": rho,
            "edge_confidence": conf,
            "edge_curvature": curv,
        })

edges_df = pd.DataFrame(edge_rows).drop_duplicates(subset=["source", "target"])
edges_df.to_csv(RESULTS_DIR / "25_residual_manifold_edges.csv", index=False)

print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())
edges_df.head()

## 5. Figure — residual kNN transport graph

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

# draw edges
for u, v, data in G.edges(data=True):
    x1, y1 = G.nodes[u]["PC1"], G.nodes[u]["PC2"]
    x2, y2 = G.nodes[v]["PC1"], G.nodes[v]["PC2"]
    ax.plot([x1, x2], [y1, y2], color="0.75", linewidth=1, alpha=0.6, zorder=1)

# draw points by family
for fam, sub in known.groupby("family"):
    ax.scatter(sub["PC1"], sub["PC2"], s=90, alpha=0.85, label=fam, zorder=3)
    for _, r in sub.iterrows():
        ax.text(r["PC1"], r["PC2"], f"N={int(r['N'])}", fontsize=8, alpha=0.8)

# centroids
for fam, r in centroids.iterrows():
    ax.scatter(r["PC1"], r["PC2"], marker="*", s=400, color="black", zorder=5)
    ax.text(r["PC1"], r["PC2"], f" {fam}", weight="bold", fontsize=10, zorder=6)

ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_title("Residual manifold kNN transport graph")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
ax.legend(loc="best")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_residual_knn_graph.png", dpi=180)
plt.show()

## 6. Family-to-family geodesic distances

Compute shortest paths between family centroids using nearest observed manifold nodes.

In [ ]:
# representative node nearest each centroid
representative = {}
for fam, cxy in centroids.iterrows():
    sub = known[known["family"] == fam]
    sub_coords = sub[["PC1", "PC2"]].to_numpy()
    d = np.linalg.norm(sub_coords - cxy.to_numpy(), axis=1)
    representative[fam] = int(sub.index[np.argmin(d)])

geo_rows = []
path_records = {}

for f1 in families:
    for f2 in families:
        n1, n2 = representative[f1], representative[f2]
        if f1 == f2:
            length = 0.0
            path = [n1]
        else:
            try:
                path = nx.shortest_path(G, source=n1, target=n2, weight="weight")
                length = nx.shortest_path_length(G, source=n1, target=n2, weight="weight")
            except nx.NetworkXNoPath:
                path = []
                length = np.nan

        if path:
            path_df = known.loc[path]
            avg_density = path_df["density_proxy"].mean()
            avg_conf = path_df["confidence_proxy"].mean()
            avg_curv = path_df["curvature_proxy"].mean()
            bridge_score = (avg_density * avg_conf) / (length + 1e-9) if f1 != f2 else np.nan
        else:
            avg_density = avg_conf = avg_curv = bridge_score = np.nan

        geo_rows.append({
            "family_i": f1,
            "family_j": f2,
            "geodesic_transport_cost": length,
            "path_node_count": len(path),
            "path_avg_density": avg_density,
            "path_avg_confidence": avg_conf,
            "path_avg_curvature": avg_curv,
            "bridge_score": bridge_score,
        })

        path_records[f"{f1} -> {f2}"] = [int(x) for x in path]

geo_df = pd.DataFrame(geo_rows)
geo_matrix = geo_df.pivot(index="family_i", columns="family_j", values="geodesic_transport_cost")
bridge_matrix = geo_df.pivot(index="family_i", columns="family_j", values="bridge_score")

geo_df.to_csv(RESULTS_DIR / "25_geodesic_transport_summary.csv", index=False)
geo_matrix.to_csv(RESULTS_DIR / "25_geodesic_distance_matrix.csv")
bridge_matrix.to_csv(RESULTS_DIR / "25_manifold_bridge_scores.csv")

with open(RESULTS_DIR / "25_family_bridge_paths.json", "w") as f:
    json.dump(path_records, f, indent=2)

geo_matrix

## 7. Figure — geodesic transport paths between family centroids

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

# base edges
for u, v, data in G.edges(data=True):
    x1, y1 = G.nodes[u]["PC1"], G.nodes[u]["PC2"]
    x2, y2 = G.nodes[v]["PC1"], G.nodes[v]["PC2"]
    ax.plot([x1, x2], [y1, y2], color="0.85", linewidth=0.8, alpha=0.5)

# selected bridges
selected_pairs = []
if len(families) >= 2:
    # consecutive centroid order along PC1
    ordered = centroids.sort_values("PC1").index.tolist()
    selected_pairs = list(zip(ordered[:-1], ordered[1:]))

for f1, f2 in selected_pairs:
    key = f"{f1} -> {f2}"
    path = path_records.get(key, [])
    if len(path) >= 2:
        p = known.loc[path]
        ax.plot(p["PC1"], p["PC2"], linewidth=3, alpha=0.9, label=f"{f1} → {f2}", zorder=4)
        ax.scatter(p["PC1"], p["PC2"], s=80, zorder=5)

# all points
for fam, sub in known.groupby("family"):
    ax.scatter(sub["PC1"], sub["PC2"], s=60, alpha=0.65, zorder=3)

# centroids
for fam, r in centroids.iterrows():
    ax.scatter(r["PC1"], r["PC2"], marker="*", s=450, color="black", zorder=6)
    ax.text(r["PC1"], r["PC2"], f" {fam}", weight="bold", fontsize=10)

ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_title("Family-to-family geodesic transport paths")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_geodesic_transport_paths.png", dpi=180)
plt.show()

## 8. Figure — geodesic cost matrix and bridgeability matrix

In [ ]:
def heatmap_matrix(mat, title, filename, cmap="viridis"):
    fig, ax = plt.subplots(figsize=(9, 7))
    M = mat.to_numpy(dtype=float)
    im = ax.imshow(M, cmap=cmap)
    ax.set_xticks(range(len(mat.columns)))
    ax.set_yticks(range(len(mat.index)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticklabels(mat.index)

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            val = M[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="black", fontsize=9)

    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=180)
    plt.show()

heatmap_matrix(geo_matrix, "Geodesic transport cost matrix", "25_geodesic_cost_matrix.png")
heatmap_matrix(bridge_matrix.fillna(0), "Universality bridgeability matrix", "25_bridge_matrix_heatmap.png")

## 9. Bottleneck region detection

Bottlenecks are nodes with:

- low density,
- low confidence,
- high curvature,
- high betweenness centrality.

The combined bottleneck score is normalized to `[0, 1]`.

In [ ]:
bet = nx.betweenness_centrality(G, weight="weight", normalized=True)
known["graph_betweenness"] = known.index.map(bet).fillna(0.0)

def minmax(x):
    x = np.asarray(x, dtype=float)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x) + 1e-12)

known["bottleneck_score"] = (
    0.30 * (1 - known["density_proxy"]) +
    0.25 * (1 - known["confidence_proxy"]) +
    0.25 * known["curvature_proxy"] +
    0.20 * minmax(known["graph_betweenness"])
)

known["bottleneck_score"] = minmax(known["bottleneck_score"])

bottlenecks = known.sort_values("bottleneck_score", ascending=False).head(max(5, len(known)//4))
bottlenecks.to_csv(RESULTS_DIR / "25_bottleneck_regions.csv", index=False)
known.to_csv(RESULTS_DIR / "25_transport_node_scores.csv", index=False)

bottlenecks[["family", "N", "PC1", "PC2", "density_proxy", "confidence_proxy", "curvature_proxy", "graph_betweenness", "bottleneck_score"]]

## 10. Figure — bottleneck regions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

# base graph
for u, v, data in G.edges(data=True):
    x1, y1 = G.nodes[u]["PC1"], G.nodes[u]["PC2"]
    x2, y2 = G.nodes[v]["PC1"], G.nodes[v]["PC2"]
    ax.plot([x1, x2], [y1, y2], color="0.85", linewidth=1, alpha=0.5)

sc = ax.scatter(
    known["PC1"], known["PC2"],
    s=100 + 600 * known["bottleneck_score"],
    c=known["bottleneck_score"],
    alpha=0.85,
    zorder=4
)

for _, r in bottlenecks.iterrows():
    ax.text(r["PC1"], r["PC2"], f"{r['family']}\nN={int(r['N'])}", fontsize=8, weight="bold")

for fam, r in centroids.iterrows():
    ax.scatter(r["PC1"], r["PC2"], marker="*", s=400, color="black", zorder=6)

ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_title("Residual transport bottleneck regions")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
plt.colorbar(sc, ax=ax, label="bottleneck score")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_bottleneck_regions.png", dpi=180)
plt.show()

## 11. Transport vector field

Estimate a local vector field by combining:

- family trajectory tangents,
- density gradient,
- confidence gradient.

This is not a physical velocity field. It is a residual manifold flow proxy.

In [ ]:
# Trajectory tangent vectors at known points
known["tx"] = 0.0
known["ty"] = 0.0

for fam, sub in known.sort_values("N").groupby("family"):
    idx = sub.index.to_list()
    pts = sub[["PC1", "PC2"]].to_numpy()
    tang = np.zeros_like(pts)
    if len(pts) == 1:
        tang[:] = 0
    elif len(pts) == 2:
        tang[0] = pts[1] - pts[0]
        tang[1] = pts[1] - pts[0]
    else:
        tang[0] = pts[1] - pts[0]
        tang[-1] = pts[-1] - pts[-2]
        for i in range(1, len(pts)-1):
            tang[i] = 0.5 * (pts[i+1] - pts[i-1])

    norms = np.linalg.norm(tang, axis=1, keepdims=True)
    tang = tang / (norms + 1e-12)

    known.loc[idx, "tx"] = tang[:, 0]
    known.loc[idx, "ty"] = tang[:, 1]

# Grid support
pad = 0.75
xmin, xmax = known["PC1"].min() - pad, known["PC1"].max() + pad
ymin, ymax = known["PC2"].min() - pad, known["PC2"].max() + pad

gx = np.linspace(xmin, xmax, 40)
gy = np.linspace(ymin, ymax, 40)
GX, GY = np.meshgrid(gx, gy)
grid_points = np.column_stack([GX.ravel(), GY.ravel()])

# interpolate density/confidence/tangent
points = known[["PC1", "PC2"]].to_numpy()
rho_grid = griddata(points, known["density_proxy"], grid_points, method="linear")
conf_grid = griddata(points, known["confidence_proxy"], grid_points, method="linear")
tx_grid = griddata(points, known["tx"], grid_points, method="linear")
ty_grid = griddata(points, known["ty"], grid_points, method="linear")

# fallback nearest for NaNs
rho_near = griddata(points, known["density_proxy"], grid_points, method="nearest")
conf_near = griddata(points, known["confidence_proxy"], grid_points, method="nearest")
tx_near = griddata(points, known["tx"], grid_points, method="nearest")
ty_near = griddata(points, known["ty"], grid_points, method="nearest")

rho_grid = np.where(np.isfinite(rho_grid), rho_grid, rho_near).reshape(GX.shape)
conf_grid = np.where(np.isfinite(conf_grid), conf_grid, conf_near).reshape(GX.shape)
tx_grid = np.where(np.isfinite(tx_grid), tx_grid, tx_near).reshape(GX.shape)
ty_grid = np.where(np.isfinite(ty_grid), ty_grid, ty_near).reshape(GX.shape)

# gradients
drho_dy, drho_dx = np.gradient(rho_grid, gy, gx)
dconf_dy, dconf_dx = np.gradient(conf_grid, gy, gx)

alpha, beta, gamma = 0.35, 0.25, 0.40
FX = alpha * drho_dx + beta * dconf_dx + gamma * tx_grid
FY = alpha * drho_dy + beta * dconf_dy + gamma * ty_grid

norm = np.sqrt(FX**2 + FY**2)
FXn = FX / (norm + 1e-12)
FYn = FY / (norm + 1e-12)

flow_samples = pd.DataFrame({
    "PC1": GX.ravel(),
    "PC2": GY.ravel(),
    "density_proxy": rho_grid.ravel(),
    "confidence_proxy": conf_grid.ravel(),
    "flow_x": FXn.ravel(),
    "flow_y": FYn.ravel(),
    "flow_magnitude": norm.ravel(),
})
flow_samples.to_csv(RESULTS_DIR / "25_flow_field_samples.csv", index=False)

flow_samples.head()

## 12. Figure — residual transport vector field

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

Z = rho_grid * conf_grid
cf = ax.contourf(GX, GY, Z, levels=25, alpha=0.75)
ax.quiver(GX[::3, ::3], GY[::3, ::3], FXn[::3, ::3], FYn[::3, ::3], alpha=0.75)

for fam, sub in known.groupby("family"):
    ax.plot(sub.sort_values("N")["PC1"], sub.sort_values("N")["PC2"], marker="o", linewidth=2, label=fam)

for fam, r in centroids.iterrows():
    ax.scatter(r["PC1"], r["PC2"], marker="*", s=350, color="black", zorder=6)
    ax.text(r["PC1"], r["PC2"], f" {fam}", weight="bold", fontsize=9)

ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_title("Residual transport vector field\n(background = density × confidence)")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
plt.colorbar(cf, ax=ax, label="density × confidence")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_transport_vector_field.png", dpi=180)
plt.show()

## 13. Curvature-weighted transport cost by family

Summarizes each family trajectory as a path through residual manifold space.

In [ ]:
trajectory_rows = []

for fam, sub in known.sort_values("N").groupby("family"):
    pts = sub[["PC1", "PC2"]].to_numpy()
    Ns = sub["N"].to_numpy()

    if len(pts) < 2:
        length = 0.0
        curvature_sum = 0.0
        endpoint = 0.0
    else:
        steps = np.linalg.norm(np.diff(pts, axis=0), axis=1)
        length = float(steps.sum())
        endpoint = float(np.linalg.norm(pts[-1] - pts[0]))

        curvature_vals = []
        if len(pts) >= 3:
            for i in range(1, len(pts)-1):
                curvature_vals.append(np.linalg.norm(pts[i+1] - 2*pts[i] + pts[i-1]))
        curvature_sum = float(np.sum(curvature_vals)) if curvature_vals else 0.0

    avg_density = float(sub["density_proxy"].mean())
    avg_conf = float(sub["confidence_proxy"].mean())
    avg_bottleneck = float(sub["bottleneck_score"].mean())
    weighted_cost = float(length * (1 + curvature_sum) * (1 + avg_bottleneck) / (avg_density * avg_conf + 1e-9))

    trajectory_rows.append({
        "family": fam,
        "trajectory_length": length,
        "endpoint_distance": endpoint,
        "curvature_sum": curvature_sum,
        "avg_density": avg_density,
        "avg_confidence": avg_conf,
        "avg_bottleneck": avg_bottleneck,
        "curvature_weighted_transport_cost": weighted_cost,
    })

traj_df = pd.DataFrame(trajectory_rows).sort_values("curvature_weighted_transport_cost", ascending=False)
traj_df.to_csv(RESULTS_DIR / "25_trajectory_transport_costs.csv", index=False)
traj_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = traj_df.sort_values("curvature_weighted_transport_cost")
ax.barh(plot_df["family"], plot_df["curvature_weighted_transport_cost"])
ax.set_title("Curvature-weighted residual transport cost")
ax.set_xlabel("transport cost")
ax.set_ylabel("family")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_transport_cost_by_family.png", dpi=180)
plt.show()

## 14. Transport phase diagrams

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    known["confidence_proxy"],
    known["density_proxy"],
    s=120 + 600 * known["curvature_proxy"],
    c=known["bottleneck_score"],
    alpha=0.85,
)

for _, r in known.iterrows():
    ax.text(r["confidence_proxy"], r["density_proxy"], f"{r['family']} N={int(r['N'])}", fontsize=7, alpha=0.75)

ax.set_title("Transport phase diagram\n(size = curvature, color = bottleneck)")
ax.set_xlabel("boundary confidence proxy")
ax.set_ylabel("density support proxy")
plt.colorbar(sc, ax=ax, label="bottleneck score")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_transport_phase_diagram.png", dpi=180)
plt.show()

## 15. Interpretation summary

This cell writes a markdown summary suitable for `docs/` or paper notes.

In [ ]:
summary_md = f'''# Notebook 25 Summary — Geodesic Transport and Residual Flow

Notebook 25 converts the residual universality manifold into a weighted transport geometry.

## Inputs

Loaded known manifold data from:

`{known_path}`

## Main results

- Built a kNN graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.
- Computed family-to-family geodesic transport costs.
- Estimated density support, boundary confidence, and local curvature proxies.
- Identified bottleneck regions where transport has low density, low confidence, high curvature, or high graph betweenness.
- Computed family bridgeability scores.
- Estimated a residual transport vector field from trajectory tangents and density/confidence gradients.

## Highest-cost family trajectories

{traj_df.head(5).to_markdown(index=False)}

## Interpretation

Residual graph families can be studied as paths through a shared residual manifold.  
Geodesic transport exposes which topology families are nearby under residual structure, while bottleneck scores identify unstable transition regions.

This supports a continuous universality-transport framing rather than a purely discrete topology-classification framing.
'''

(RESULTS_DIR / "25_summary.md").write_text(summary_md)
print(summary_md)

## 16. Export manifest and optional download

This final cell zips Notebook 25 outputs.

In Colab, it also opens a download prompt.

In [ ]:
manifest = {
    "notebook": "25_geodesic_transport_and_residual_flow.ipynb",
    "created_outputs": {
        "figures": sorted([p.name for p in FIGURES_DIR.glob("25_*.png")]),
        "results": sorted([p.name for p in RESULTS_DIR.glob("25_*")]),
    },
    "source_file": str(known_path),
}

manifest_path = EXPORTS_DIR / "25_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / "25_geodesic_transport_and_residual_flow_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in FIGURES_DIR.glob("25_*.png"):
        z.write(p, arcname=f"figures/{p.name}")
    for p in RESULTS_DIR.glob("25_*"):
        z.write(p, arcname=f"results/{p.name}")
    z.write(manifest_path, arcname="exports/25_manifest.json")

print("Wrote:", zip_path)
print("Zip size MB:", zip_path.stat().st_size / 1e6)

# Optional Colab download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Colab download skipped. Download manually from:", zip_path)